In [1]:
# ====================================================IMPORT=================================================================

from datasets import DatasetDict, load_dataset, load_from_disk, Dataset

import os
import warnings
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

import time
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore")


#from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import pandas as pd
import sys
sys.path.append("C:\\Users\\Rumble\\Documents\\Mestrado PUC\\silver_clone\\mt_luxembourgish")
import os
#from datasets import Dataset
import argparse

from utils.utils_train import pre_process, create_prompt, print_trainable_parameters, create_prompt_gemma
from transformers import TrainingArguments, DataCollatorForSeq2Seq

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
import torch
from peft import get_peft_model, LoraConfig
from transformers import logging as transformers_logging
from accelerate import Accelerator

import warnings

warnings.simplefilter("ignore")
transformers_logging.set_verbosity_error()


c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.5.1+cu121
12.1
True
NVIDIA GeForce RTX 2060


In [2]:
# ========================== CMD Argument Parser ==========================
from types import SimpleNamespace

args = SimpleNamespace(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    src_lng="Tupi",
    tgt_lng="Português",
    num_train_epochs=1,
    learning_rate=1e-5,
    project_root=r"C:\Users\Rumble\Documents\Mestrado PUC\silver_clone\mt_luxembourgish",
    training_dataset_path = r"C:\Users\Rumble\Documents\Mestrado PUC\script\Dataset_tupi\dataset_augmentation_v2.json",
    model_path="Qwen/Qwen2.5-3B-Instruct",
    resume_from_checkpoint=False,
    resume_checkpoint_path=None,
    r=256,
    is_peft=True,
    is_unsloth=True,
    is_train_response_only=False
)

print("Arguments passed:")
print(f"Train Batch Size: {args.per_device_train_batch_size}")
print(f"Eval Batch Size: {args.per_device_eval_batch_size}")
print(f"Number of Epochs: {args.num_train_epochs}")
print(f"Learning Rate: {args.learning_rate}")
print(f"Project Root: {args.project_root}")
print(f"Training Dataset Path: {args.training_dataset_path}")
print(f"Model path: {args.model_path}")
print(f"tgt_lng: {args.tgt_lng}")
print(f"src_lng: {args.src_lng}")
print(f"Resume from checkpoint: {args.resume_from_checkpoint}")
print(f"Resume checkpoint path: {args.resume_checkpoint_path}")
print(f"r: {args.r}")
print(f"is_peft: {args.is_peft}")
print(f"is_unsloth: {args.is_unsloth}")
print(f"is_train_response_only: {args.is_train_response_only}")

Arguments passed:
Train Batch Size: 1
Eval Batch Size: 1
Number of Epochs: 1
Learning Rate: 1e-05
Project Root: C:\Users\Rumble\Documents\Mestrado PUC\silver_clone\mt_luxembourgish
Training Dataset Path: C:\Users\Rumble\Documents\Mestrado PUC\script\Dataset_tupi\dataset_augmentation_v2.json
Model path: Qwen/Qwen2.5-3B-Instruct
tgt_lng: Português
src_lng: Tupi
Resume from checkpoint: False
Resume checkpoint path: None
r: 256
is_peft: True
is_unsloth: True
is_train_response_only: False


In [3]:
learning_rate = args.learning_rate # Learning rate for the optimizer
per_device_train_batch_size = args.per_device_train_batch_size  # Batch size for training per device
per_device_eval_batch_size = args.per_device_eval_batch_size  # Batch size for evaluation per device
num_train_epochs = args.num_train_epochs  # Number of epochs for training
training_dataset_path = args.training_dataset_path
project_root = args.project_root
model_path = args.model_path
resume_from_checkpoint = args.resume_from_checkpoint
resume_checkpoint_path = args.resume_checkpoint_path
src_lng = args.src_lng
tgt_lng = args.tgt_lng
r = args.r
is_peft = args.is_peft
is_unsloth = args.is_unsloth
is_train_response_only = args.is_train_response_only

if is_unsloth:
    #from unsloth import is_bfloat16_supported
    #from unsloth.chat_templates import train_on_responses_only
    #from unsloth import FastLanguageModel
    #fp16 = not is_bfloat16_supported()
    #bf16 = is_bfloat16_supported(),
    fp16 = False
    bf16 = False
else:
    fp16 = False
    bf16 = False



#model_name = model_path.split("/")[-1]
train_ratio = 1.0  # Number of samples to be used for training and evaluation
warmup_ratio = 0.5
logging_steps = 1000
evaluation_strategy="steps"
save_strategy="epoch"
eval_steps=1000
max_grad_norm = 0.3
MAX_LEN = 128
weight_decay = 0.01
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.
train_seed = 42

if is_peft:
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj",]
    lora_alpha = 8
    lora_dropout = 0
    random_state = 42

current = time.time()
formatted_time = time.strftime("%m_%d_%H_%M", time.localtime(current))

if resume_from_checkpoint:
    output_dir = resume_checkpoint_path

else:

    input_file_name = os.path.splitext(
        os.path.basename(training_dataset_path)
    )[0]

    if is_peft:

        output_dir = (
            f"logs/peft_{r}_{src_lng[:2]}_{tgt_lng[:2]}/"
            f"fit_{formatted_time}_{train_ratio}_{input_file_name}"
        )

    else:

        output_dir = (
            f"logs/full_{src_lng[:2]}_{tgt_lng[:2]}/"
            f"fit_{formatted_time}_{train_ratio}_{input_file_name}"
        )

if resume_from_checkpoint and resume_checkpoint_path is None:
    raise ValueError("Please provide a checkpoint path to resume training from")

In [4]:
# ========================== dataset preparation ==========================

#train_dataset_path = os.path.abspath(os.path.join(project_root, training_dataset_path))
train_dataset_path = os.path.abspath(os.path.join(training_dataset_path))
sys.path.append(project_root)

train_dataset_df = pd.read_json(train_dataset_path) #, lines=True)
pre_processed_dataset_df = pre_process(train_dataset_df)

if not isinstance(pre_processed_dataset_df, pd.DataFrame):
    raise TypeError("data_preprocess should return a pandas DataFrame.")

dataset = Dataset.from_pandas(pre_processed_dataset_df)

dataset = dataset.shuffle(seed=42)
dataset = dataset.map(
    lambda x, idx: {
        "split": "train" if idx < int(0.7 * len(dataset)) else "val"
    },
    with_indices=True
)

# Filter by split
train_dataset = dataset.filter(lambda x: x["split"] == "train")
val_dataset = dataset.filter(lambda x: x["split"] == "val")

# Select subset
train_dataset = train_dataset.select(range(int(len(train_dataset) * train_ratio)))
val_dataset = val_dataset.select(range(int(len(val_dataset) * train_ratio)))  # Avoid out-of-range error

# Rename columns
train_dataset = train_dataset.rename_columns({
    "input": "Tupi",
    "output": "Português",
})

val_dataset = val_dataset.rename_columns({
    "input": "Tupi",
    "output": "Português",
})

val_dataset.to_json("val_dataset.json")
train_dataset.to_json("train_dataset.json")

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

train_dataset = train_dataset.map(
    lambda sample: {
        "full_prompt": create_prompt(sample, src_lng=src_lng, tgt_lng=tgt_lng, mode="train", tokenizer=tokenizer)["full_prompt"]
    }
).select_columns(["full_prompt"])

val_dataset = val_dataset.map(
    lambda sample: {
        "full_prompt": create_prompt(sample, src_lng=src_lng, tgt_lng=tgt_lng, mode="train", tokenizer=tokenizer)["full_prompt"]
    }
).select_columns(["full_prompt"])

print (train_dataset["full_prompt"][0])

def tokenize_function(examples):
    return tokenizer(
        examples["full_prompt"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["full_prompt"])
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["full_prompt"])

print("Dataset tokenized:")
print(tokenized_train_dataset[0])

Length of inputs before: 2012
Length of inputs after: 2012
Removed rows: 0


Map: 100%|██████████| 604/604 [00:00<00:00, 6812.87 examples/s]


<|im_start|>system
Você é um assistente de IA muito útil para traduções.<|im_end|>
<|im_start|>user
Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

kunhãmukuetá îkó ka'ape<|im_end|>
<|im_start|>assistant
moças vivem na mata<|im_end|><|im_end|>



Map: 100%|██████████| 604/604 [00:00<00:00, 12567.27 examples/s]

Dataset tokenized:
{'input_ids': [151644, 8948, 198, 69286, 3958, 4443, 7789, 6817, 409, 43090, 33750, 137741, 3348, 4685, 84, 15249, 13, 151645, 198, 151644, 872, 198, 42834, 84, 4360, 297, 93072, 68, 32025, 976, 96097, 72, 3348, 22234, 84, 36830, 13, 57649, 18409, 4284, 64066, 80367, 2782, 5908, 136836, 24870, 3449, 4942, 382, 74, 359, 71, 3202, 76, 3101, 13807, 1953, 14364, 74, 1794, 16502, 6, 2027, 151645, 198, 151644, 77091, 198, 6355, 45278, 17950, 336, 4317, 96193, 151645, 151645, 198, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
if is_unsloth:
    bnb_config = BitsAndBytesConfig(
    load_in_4bit=load_in_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
    model_path, #model_name,
    trust_remote_code=True
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
    model_path, #model_name,
    quantization_config=bnb_config if load_in_4bit else None,
    #device_map="auto",
    trust_remote_code=True,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    if load_in_4bit:
        model = prepare_model_for_kbit_training(model)

    if is_peft:
        lora_config = LoraConfig(
            r=r,
            target_modules=target_modules,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            #task_type="CAUSAL_LM",
            use_rslora=False,
            #random_state = random_state,
            #loftq_config = None,
        )

        model = get_peft_model(model, lora_config)
# Using transformer huggingface   
else:
    model = AutoModelForCausalLM.from_pretrained(model_path)
    model.config.use_cache = False
    if is_peft:
        lora_config = LoraConfig(
            r=r, 
            target_modules=target_modules, 
            lora_alpha=lora_alpha, 
            lora_dropout=lora_dropout, 
            bias="none", 
            random_state=random_state,
            use_rslora=False,
            loftq_config=None  # And without LoftQ
        )
        model = get_peft_model(model, lora_config)

print (model)
print(print_trainable_parameters(model))

Loading checkpoint shards: 100%|██████████| 2/2 [01:04<00:00, 32.34s/it]


PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (base_la

In [6]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name}, Shape: {param.shape}, Trainable parameters: {param.numel()}")
    else:
        print(f"Layer: {name}, Shape: {param.shape}, Non-trainable parameters: {param.numel()}")

Layer: base_model.model.model.embed_tokens.weight, Shape: torch.Size([151936, 2048]), Non-trainable parameters: 311164928
Layer: base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight, Shape: torch.Size([2048, 2048]), Non-trainable parameters: 4194304
Layer: base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias, Shape: torch.Size([2048]), Non-trainable parameters: 2048
Layer: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight, Shape: torch.Size([128, 2048]), Trainable parameters: 262144
Layer: base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight, Shape: torch.Size([2048, 128]), Trainable parameters: 262144
Layer: base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight, Shape: torch.Size([256, 2048]), Non-trainable parameters: 524288
Layer: base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias, Shape: torch.Size([256]), Non-trainable parameters: 256
Layer: base_model.model.model.layers.0.self_attn.k_proj.

In [ ]:
def train_ddp_accelerate_sft():
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        warmup_ratio=warmup_ratio,
        evaluation_strategy=evaluation_strategy,
        save_strategy=save_strategy,
        logging_steps=logging_steps,
        eval_steps=eval_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        fp16 = fp16,
        bf16 =bf16,
        max_grad_norm=max_grad_norm,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="tensorboard",
        remove_unused_columns=False,
        disable_tqdm=False,
        seed = train_seed,
        ddp_find_unused_parameters=False, 
        dataloader_num_workers=2,
        dataset_text_field = "full_prompt",
        max_seq_length = MAX_LEN,
        dataset_num_proc = 2,
        packing = False, # Can make training 5x faster for short sequences.
        # load_best_model_at_end=True,
    )

    #response_template = "<|im_start|>assistant\n"
    #data_collator = DataCollatorForLanguageModeling(
    #    tokenizer=tokenizer,
    #    mlm=False
    #)

    trainer = SFTTrainer(
        model=model,
        #train_dataset=train_dataset,
        #eval_dataset=val_dataset,
        train_dataset = tokenized_train_dataset,
        eval_dataset = tokenized_val_dataset,
        tokenizer=tokenizer,
        #data_collator=data_collator,
        args=training_args
    )


    if is_train_response_only:
        if "gemma" not in model_path and is_unsloth:
            trainer = train_on_responses_only(
                trainer,
                instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
                response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
            )
            

    trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    print("Finished training SFT.")
    return trainer_stats

In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 2060. Max memory = 6.0 GB.
0.0 GB of memory reserved.


In [9]:
print(train_dataset.column_names)

print(type(train_dataset[0]["full_prompt"]))
print(train_dataset[0]["full_prompt"][:300])

['full_prompt']
<class 'str'>
<|im_start|>system
Você é um assistente de IA muito útil para traduções.<|im_end|>
<|im_start|>user
Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

kunhãmukuetá îkó ka'ape<|im_end|>
<|im_start|>assistant
moças vivem na mata<|im_end|><|im_e


In [10]:
trainer_stats = None

def main():
    trainer_stats = train_ddp_accelerate_sft()
    return trainer_stats

if __name__ == "__main__":
    trainer_stats = main()

 24%|██▎       | 1000/4224 [3:42:34<11:50:12, 13.22s/it]

{'loss': 2.7059, 'grad_norm': 0.7248430252075195, 'learning_rate': 4.734848484848486e-06, 'epoch': 0.71}


                                                        
 24%|██▎       | 1000/4224 [4:23:23<11:50:12, 13.22s/it]

{'eval_runtime': 2448.5419, 'eval_samples_per_second': 0.247, 'eval_steps_per_second': 0.247, 'epoch': 0.71}


 47%|████▋     | 2000/4224 [8:06:20<8:12:26, 13.29s/it]   

...

{'loss': 0.6574, 'grad_norm': 1.2094190120697021, 'learning_rate': 2.7499590642665773e-07, 'epoch': 2.84}


                                                      
 95%|█████████▍| 4000/4224 [17:33:51<50:58, 13.65s/it]

{'eval_runtime': 2440.7918, 'eval_samples_per_second': 0.247, 'eval_steps_per_second': 0.247, 'epoch': 2.84}


100%|██████████| 4224/4224 [18:24:14<00:00, 15.69s/it]    

{'train_runtime': 66254.7913, 'train_samples_per_second': 0.064, 'train_steps_per_second': 0.064, 'train_loss': 1.27376681024378, 'epoch': 3.0}
Finished training SFT.


In [11]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

66254.7913 seconds used for training.
1104.25 minutes used for training.
Peak reserved memory = 17.684 GB.
Peak reserved memory for training = 17.684 GB.
Peak reserved memory % of max memory = 294.733 %.
Peak reserved memory for training % of max memory = 294.733 %.
